In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from google.colab import drive
if os.path.exists('/content/inputs.json.gz'):
    D = '/content'
else:
    drive.mount('/content/drive')
    D = '/content/drive/MyDrive/temporal-ood-ddi'
os.makedirs(f'{D}/runs/preds', exist_ok=True)
print(D)

In [ ]:
!mkdir -p /content/w && tar xzf {D}/ddiood_code.tgz -C /content/w 2>/dev/null; cp {D}/inputs.json.gz /content/w/
!pip -q install rank_bm25 bitsandbytes accelerate

In [ ]:
import base64, gzip, hashlib, json
DELTA = '...'
d = json.loads(gzip.decompress(base64.b64decode(DELTA)))
b = json.load(gzip.open('/content/w/inputs.json.gz', 'rt'))
for v, ch in d['retrieval'].items():
    b['retrieval'][v].update(ch)
for v, ks in d['drop_retrieval'].items():
    for k in ks:
        b['retrieval'][v].pop(k, None)
b['chunk_text'].update(d['chunk_text'])
for k in d['drop_chunks']:
    b['chunk_text'].pop(k, None)
h = hashlib.sha256(json.dumps({'retrieval': b['retrieval'], 'chunk_text': b['chunk_text']}, sort_keys=True).encode()).hexdigest()
assert h == '8ff6794627912f5bbeb0d35f26beefdd39ebc51141fad88e13cbf9c94134155b', h
with gzip.open('/content/w/inputs.json.gz', 'wt') as f:
    json.dump(b, f)
print('corrected bundle ready', h)

In [ ]:
JOBS = [
 "llm-biomistral-names",
 "llm-qwen-names",
 "llm-biomistral-rag",
 "llm-qwen-rag"
]

In [ ]:
%cd /content/w
!python -m ddiood.gpu_jobs --bundle inputs.json.gz --out {D}/runs --only {' '.join(JOBS)} 2>&1 | tee -a {D}/runs/E5_llm_baselines.log | grep --line-buffered -E '^(start|done|skip)|Error|Traceback'

In [ ]:
import os
done = sorted(j for j in JOBS if os.path.exists(f'{D}/runs/preds/{j}.json'))
print(f'{len(done)}/{len(JOBS)} jobs finished')
print('missing:', [j for j in JOBS if j not in done])